In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc numpy qiskit-addon-utils
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

<details>
<summary><b>Paketversionen</b></summary>

Der Code auf dieser Seite wurde mit den folgenden Anforderungen entwickelt.
Wir empfehlen, diese oder neuere Versionen zu verwenden.

```
qiskit[all]~=2.3.0
qiskit-ibm-runtime~=0.43.1
qiskit-addon-utils~=0.3.0
```
</details>

Das Paket Qiskit-Addon-Hilfsprogramme ist eine Sammlung von Funktionalitäten zur Ergänzung von Workflows, die ein oder mehrere Qiskit-Addons umfassen. Das Paket enthält beispielsweise Funktionen zum Erstellen von Hamiltonoperatoren, zum Generieren von Trotter-Zeitentwicklungs-Circuits und zum Aufteilen und Kombinieren von Quantum-Circuits.

## Installation

Es gibt zwei Möglichkeiten, die Qiskit-Addon-Hilfsprogramme zu installieren: über PyPI oder durch Kompilieren aus dem Quellcode. Es wird empfohlen, diese Pakete in einer [virtuellen Umgebung](https://docs.python.org/3.10/tutorial/venv.html) zu installieren, um eine Trennung der Paketabhängigkeiten sicherzustellen.

### Installation über PyPI

Die einfachste Methode zur Installation des Pakets Qiskit-Addon-Hilfsprogramme ist über PyPI.

In [1]:
from qiskit_ibm_runtime.fake_provider import FakeSherbrooke
from qiskit import QuantumCircuit
from qiskit_addon_utils.coloring import auto_color_edges
from qiskit_addon_utils.slicing import combine_slices, slice_by_depth
from collections import defaultdict

backend = FakeSherbrooke()
coupling_map = backend.coupling_map

# Create naive circuit
circuit = QuantumCircuit(backend.num_qubits)
for edge in coupling_map.graph.edge_list():
    circuit.cz(edge[0], edge[1])


# Color the edges of the coupling map
coloring = auto_color_edges(coupling_map)
circuit_with_coloring = QuantumCircuit(backend.num_qubits)

# Make a reverse coloring dict in order to make the circuit
color_to_edge = defaultdict(list)
for edge, color in coloring.items():
    color_to_edge[color].append(edge)

# Place edges in order of color
for edges in color_to_edge.values():
    for edge in edges:
        circuit_with_coloring.cz(edge[0], edge[1])

print(f"The circuit without using edge coloring has depth: {circuit.depth()}")
print(
    f"The circuit using edge coloring has depth: {circuit_with_coloring.depth()}"
)

The circuit without using edge coloring has depth: 37
The circuit using edge coloring has depth: 3


### Installation aus dem Quellcode
<details>
<summary>
Hier klicken, um zu erfahren, wie du dieses Paket manuell installierst.
</summary>

Wenn du zu diesem Paket beitragen möchtest oder es manuell installieren willst, klone zunächst das Repository:

In [2]:
import numpy as np
from qiskit import QuantumCircuit

num_qubits = 9
qc = QuantumCircuit(num_qubits)
qc.ry(np.pi / 4, range(num_qubits))
qubits_1 = [i for i in range(num_qubits) if i % 2 == 0]
qubits_2 = [i for i in range(num_qubits) if i % 2 == 1]
qc.cx(qubits_1[:-1], qubits_2)
qc.cx(qubits_2, qubits_1[1:])
qc.cx(qubits_1[-1], qubits_1[0])
qc.rx(np.pi / 4, range(num_qubits))
qc.rz(np.pi / 4, range(num_qubits))
qc.draw("mpl", scale=0.6)

<Image src="../docs/images/guides/qiskit-addons-utils/extracted-outputs/d5a5c53f-45d2-4cdc-86f3-221f179cdbea-0.svg" alt="Output of the previous code cell" />

und installiere das Paket über `pip`. Wenn du vorhast, die im Paket-Repository enthaltenen Tutorials auszuführen, installiere auch die Notebook-Abhängigkeiten. Wenn du im Repository entwickeln möchtest, installiere die `dev`-Abhängigkeiten.

In [3]:
# Slice circuit into partitions of depth 1
slices = slice_by_depth(qc, 1)

# Recombine slices in order to visualize the partitions together
combined_slices = combine_slices(slices, include_barriers=True)
combined_slices.draw("mpl", scale=0.6)

<Image src="../docs/images/guides/qiskit-addons-utils/extracted-outputs/6d824d97-edc6-4c88-bcfa-428731393f83-0.svg" alt="Output of the previous code cell" />

</details>

## Erste Schritte mit den Hilfsprogrammen
Das Paket `qiskit-addon-utils` enthält mehrere Module, darunter eines zur Problemgenerierung für die Simulation von Quantensystemen, eines zur Kantenfärbung für eine effizientere Gate-Platzierung in einem Quantum-Circuit sowie eines zum Circuit-Slicing, das bei der [Operator-Rückpropagation](/guides/qiskit-addons-obp) helfen kann. Die folgenden Abschnitte fassen jedes Modul zusammen. Die [API-Dokumentation](https://docs.quantum.ibm.com/api/qiskit-addon-utils) des Pakets enthält ebenfalls hilfreiche Informationen.

### Problemgenerierung
Der Inhalt des Moduls [`qiskit_addon_utils.problem_generators`](../api/qiskit-addon-utils/problem-generators) umfasst:

- Eine Funktion [`generate_xyz_hamiltonian()`](../api/qiskit-addon-utils/problem-generators#generate_xyz_hamiltonian), die eine konnektivitätsbewusste `SparsePauliOp`-Darstellung eines Ising-Typ-XYZ-Modells generiert:

$$H = \sum_{(j,k)\in E} \left(J_x X_jX_k + J_yY_jY_k + J_zZ_jZ_k\right) + \sum_{j\in V} \left(h_x X_j + h_y Y_j + h_z Z_j\right) $$
- Eine Funktion [`generate_time_evolution_circuit()`](../api/qiskit-addon-utils/problem-generators#generate_time_evolution_circuit), die einen Circuit konstruiert, der die Zeitentwicklung eines gegebenen Operators modelliert.
- Drei verschiedene [`PauliOrderStrategy`](../api/qiskit-addon-utils/problem-generators#pauliorderstrategy)-Objekte zur Aufzählung unterschiedlicher Pauli-String-Ordnungen. Dies ist besonders nützlich in Kombination mit der Kantenfärbung und kann als Argument sowohl in `generate_xyz_hamiltonian()` als auch in `generate_time_evolution_circuit()` verwendet werden.

### Kantenfärbung
Das Modul [`qiskit_addon_utils.coloring`](../api/qiskit-addon-utils/coloring) wird verwendet, um die Kanten einer Coupling Map einzufärben und diese Färbung für eine effizientere Gate-Platzierung in einem Quantum-Circuit zu nutzen. Der Zweck dieser kantengefärbten Coupling Map besteht darin, eine Menge von Kantenfarben zu finden, bei der keine zwei Kanten gleicher Farbe einen gemeinsamen Knoten teilen. Für einen QPU bedeutet das, dass Gates entlang gleichfarbiger Kanten (Qubit-Verbindungen) gleichzeitig ausgeführt werden können und der Circuit schneller abläuft.

Als kurzes Beispiel kannst du die Funktion [`auto_color_edges()`](../api/qiskit-addon-utils/coloring#auto_color_edges) verwenden, um eine Kantenfärbung für einen naiven Circuit zu erzeugen, der ein `CZGate` entlang jeder Qubit-Verbindung ausführt. Das folgende Code-Snippet verwendet die Coupling Map des `FakeSherbrooke`-Backends, erstellt diesen naiven Circuit und nutzt dann `auto_color_edges()`, um einen effizienteren äquivalenten Circuit zu erstellen.

In [4]:
from qiskit_addon_utils.slicing import slice_by_gate_types

slices = slice_by_gate_types(qc)

# Recombine slices in order to visualize the partitions together
combined_slices = combine_slices(slices, include_barriers=True)
combined_slices.draw("mpl", scale=0.6)

<Image src="../docs/images/guides/qiskit-addons-utils/extracted-outputs/d011c56e-d741-491a-8867-54952cb7b757-0.svg" alt="Output of the previous code cell" />

If your workload is designed to exploit the physical qubit connectivity of the QPU it will be run on, you can create slices based on edge coloring. The code snippet below will assign a three-coloring to circuit edges and slice the circuit with respect to the edge coloring. (Note: this only affects non-local gates. Single-qubit gates will be sliced by gate type).

In [5]:
from qiskit_addon_utils.slicing import slice_by_coloring

# Assign a color to each set of connected qubits
coloring = {}
for i in range(num_qubits - 1):
    coloring[(i, i + 1)] = i % 3
coloring[(num_qubits - 1, 0)] = 2

# Create a circuit with operations added in order of color
qc = QuantumCircuit(num_qubits)
qc.ry(np.pi / 4, range(num_qubits))
edges = [
    edge for color in range(3) for edge in coloring if coloring[edge] == color
]
for edge in edges:
    qc.cx(edge[0], edge[1])
qc.rx(np.pi / 4, range(num_qubits))
qc.rz(np.pi / 4, range(num_qubits))

# Create slices by edge color
slices = slice_by_coloring(qc, coloring=coloring)

# Recombine slices in order to visualize the partitions together
combined_slices = combine_slices(slices, include_barriers=True)
combined_slices.draw("mpl", scale=0.6)

<Image src="../docs/images/guides/qiskit-addons-utils/extracted-outputs/41290d70-e116-4bd2-b9e7-d90aeaa852aa-0.svg" alt="Output of the previous code cell" />

### Slicing
Das Modul [`qiskit-addon-utils.slicing`](../api/qiskit-addon-utils/slicing) enthält Funktionen und Transpiler-Passes zum Arbeiten mit Circuit-„Slices" – zeitartigen Partitionen eines [`QuantumCircuit`](../api/qiskit/qiskit.circuit.QuantumCircuit), die sich über alle Qubits erstrecken. Diese Slices werden hauptsächlich für die [Operator-Rückpropagation](/guides/qiskit-addons-obp-get-started) verwendet. Die vier Hauptmethoden zum Aufteilen eines Circuits sind nach Gate-Typ, Tiefe, Färbung oder [`Barrier`](../api/qiskit/circuit#barrier)-Anweisungen. Die Slicing-Funktionen geben eine Liste von [`QuantumCircuit`](../api/qiskit/qiskit.circuit.QuantumCircuit)-Objekten zurück. Aufgeteilte Circuits können mit der Funktion [`combine_slices()`](../api/qiskit-addon-utils/slicing#combine_slices) auch wieder zusammengeführt werden. Weitere Informationen findest du in der [API-Referenz](../api/qiskit-addon-utils/slicing) des Moduls.

Im Folgenden sind einige Beispiele aufgeführt, wie du diese Slices mit dem folgenden Circuit erstellen kannst:

In [6]:
qc = QuantumCircuit(num_qubits)
qc.ry(np.pi / 4, range(num_qubits))
qc.barrier()
qubits_1 = [i for i in range(num_qubits) if i % 2 == 0]
qubits_2 = [i for i in range(num_qubits) if i % 2 == 1]
qc.cx(qubits_1[:-1], qubits_2)
qc.cx(qubits_2, qubits_1[1:])
qc.cx(qubits_1[-1], qubits_1[0])
qc.barrier()
qc.rx(np.pi / 4, range(num_qubits))
qc.rz(np.pi / 4, range(num_qubits))
qc.draw("mpl", scale=0.6)

<Image src="../docs/images/guides/qiskit-addons-utils/extracted-outputs/5040a678-9d5f-4c8b-8a92-7de04c3fcfda-0.svg" alt="Output of the previous code cell" />

![Ausgabe der vorherigen Code-Zelle](../docs/images/guides/qiskit-addons-utils/extracted-outputs/d5a5c53f-45d2-4cdc-86f3-221f179cdbea-0.svg)

Falls es keine offensichtliche Möglichkeit gibt, die Struktur eines Circuits für die Operator-Rückpropagation auszunutzen, kannst du den Circuit in Slices einer bestimmten Tiefe aufteilen.

In [7]:
from qiskit_addon_utils.slicing import slice_by_barriers

slices = slice_by_barriers(qc)
slices[0].draw("mpl", scale=0.6)

<Image src="../docs/images/guides/qiskit-addons-utils/extracted-outputs/b1106c2f-711c-4d30-b91a-ea05f47598d8-0.svg" alt="Output of the previous code cell" />

In [8]:
slices[1].draw("mpl", scale=0.6)

<Image src="../docs/images/guides/qiskit-addons-utils/extracted-outputs/1afd2111-dd88-4e20-a137-f0c975e9d1bb-0.svg" alt="Output of the previous code cell" />

In [9]:
slices[2].draw("mpl", scale=0.6)

<Image src="../docs/images/guides/qiskit-addons-utils/extracted-outputs/360ab773-df81-4975-bb19-cd5f34e69b29-0.svg" alt="Output of the previous code cell" />

![Ausgabe der vorherigen Code-Zelle](../docs/images/guides/qiskit-addons-utils/extracted-outputs/41290d70-e116-4bd2-b9e7-d90aeaa852aa-0.svg)

Wenn du eine eigene Slicing-Strategie hast, kannst du stattdessen Barriers im Circuit platzieren, um die Stellen zu markieren, an denen der Circuit aufgeteilt werden soll, und die Funktion [`slice_by_barriers`](../api/qiskit-addon-utils/slicing#slice_by_barriers) verwenden.